# 02 · Experiments

Experiments are normally run with DVC so they are cached and versioned:

```bash
uv run dvc exp run -S model.name=legacy_cnn
uv run dvc exp run --queue -S 'train.optimizer.learning_rate=1e-3,3e-4' -S 'model.cnn.dropout=0.1,0.3'
uv run dvc queue start
```

This notebook compares finished experiments and inspects the current one. For
quick interactive iterations, stages can also be called directly with config
overrides (last section).

In [ ]:
# Run from the repository root so params.yaml and data/ paths resolve.
import os
from pathlib import Path

if not Path("params.yaml").exists():
    os.chdir(Path.cwd().parent)
print(Path.cwd())

In [ ]:
import json
import subprocess

import pandas as pd

raw = json.loads(subprocess.run(["dvc", "exp", "show", "--json"], capture_output=True, text=True, check=True).stdout)

def flatten(prefix, value, row):
    if isinstance(value, dict):
        for key, inner in value.items():
            flatten(f"{prefix}.{key}" if prefix else key, inner, row)
    else:
        row[prefix] = value

rows = []
for commit in raw:
    runs = [commit] + [exp for group in (commit.get("experiments") or []) for exp in group.get("revs", [])]
    for run in runs:
        data = run.get("data") or {}
        row = {"experiment": run.get("name") or run.get("rev")}
        for kind in ("params", "metrics"):
            for file_data in (data.get(kind) or {}).values():
                flatten("", file_data.get("data", {}), row)
        rows.append(row)

table = pd.DataFrame(rows).set_index("experiment")
columns = [c for c in ["model.name", "features.preprocess.representation", "features.preprocess.img_size",
                       "train.optimizer.learning_rate", "val_accuracy", "accuracy", "end_to_end_accuracy",
                       "macro_f1", "latency_ms", "params"] if c in table]
table[columns].sort_values("accuracy", ascending=False) if "accuracy" in table else table[columns]

## Training curves of the workspace run

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from asl.lab.config import load_config

config = load_config(["params.yaml"])
plots = Path(config.paths.train_dir) / "live" / "plots" / "metrics"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["accuracy", "loss"]):
    for split in ("train", "eval"):
        path = plots / split / f"{metric}.tsv"
        if path.exists():
            frame = pd.read_csv(path, sep="\t")
            ax.plot(frame["step"], frame[metric], label=split)
    ax.set_title(metric)
    ax.set_xlabel("epoch")
    ax.legend()
plt.show()

## Confusion matrix and weakest classes

In [ ]:
eval_dir = Path(config.paths.eval_dir)
confusion = pd.read_csv(eval_dir / "confusion.csv", dtype=str)
matrix = pd.crosstab(confusion["actual"], confusion["predicted"])

plt.figure(figsize=(10, 9))
plt.imshow(matrix.values, cmap="Blues")
plt.xticks(range(len(matrix.columns)), matrix.columns)
plt.yticks(range(len(matrix.index)), matrix.index)
plt.xlabel("predicted")
plt.ylabel("actual")
plt.colorbar()
plt.show()

per_class = pd.DataFrame(json.loads((eval_dir / "per_class.json").read_text())).T
per_class[per_class.index.str.len() == 1].sort_values("f1-score").head(10)

## Interactive run (without DVC)

Every stage is a function of a validated config. Overrides use the same
`key.path=value` syntax as `dvc exp run -S`. Writing to separate `paths.*`
keeps the DVC-tracked outputs untouched.

In [ ]:
from asl.lab.data import run_split
from asl.lab.evaluate import run_evaluate
from asl.lab.features import run_featurize
from asl.lab.train import run_train

scratch = load_config(["params.yaml", "asl/lab/configs/smoke.yaml"], [
    "paths.splits_dir=scratch/splits", "paths.features_dir=scratch/features",
    "paths.model_dir=scratch/model", "paths.train_dir=scratch/train", "paths.eval_dir=scratch/eval",
])
run_split(scratch)
run_featurize(scratch)
run_train(scratch)
run_evaluate(scratch)